# Pipeline scRNA-seq: scanpy + SCENIC

**Versión 1.6.1**

Corre en Google Colab y hace dos cosas: agrupa las células por tipo con scanpy, y con SCENIC averigua qué genes las controlan. Se puede correr sobre datos de ejemplo o sobre el estudio real de lupus, según lo que se elija en la celda 1.

| Modo | Datos | Qué hace |
|---|---|---|
| `"ejemplo"` | Demostración | Médula ósea humana, más SCENIC sobre el dataset `tiny` |
| `"zenodo"` | Lupus real | Datos de Jang et al., figuras del artículo, SCENIC sobre células B |

Se generan las mismas 12 gráficas en los dos modos (violines de QC, scatter, HVG, PCA, grafo de vecinos, UMAP de QC, anotación, marcadores, DE, DE completo y top-100, heatmap, trayectoria), con scanpy y SCENIC aplicados de principio a fin.

Para usarlo: en la celda 1 se pone `MODO = "ejemplo"` o `"zenodo"`, y luego *Entorno de ejecución → Ejecutar todas*. Las celdas del otro modo se saltan solas.

Dos cosas que conviene saber antes de correrlo.

En modo `"zenodo"`, por defecto se usan las 169.513 células completas del estudio (el número está confirmado en Zenodo y en la publicación de Jang et al.), y eso necesita RAM alta: Colab Pro/Pro+, o el entorno de RAM alta activado. La celda 2.1 revisa cuánta RAM tiene la sesión y se detiene con una explicación si no alcanza. Si solo se tiene el Colab gratuito, se pone un número en `N_CELLS_MAX` (celda 1) para trabajar con una muestra en vez del dataset completo.

En modo `"ejemplo"` no se va a obtener biología real. El dataset tiene 500 genes y 20 factores de transcripción, muy poco para que SCENIC valide algún regulón. Aun así el notebook completa el flujo entero hasta mostrar un UMAP de regulones, marcado `_SIN_VALIDAR`. La celda 6.7, al final, indica si el resultado es apto para entregar (solo lo es en modo zenodo, con regulones validados).


---
# 1 · Configuración

Es la única celda que se necesita tocar. Se elige el modo y se ajustan los parámetros que se quieran cambiar.


In [ ]:
# ============================================================
#   CELDA MAESTRA: elige que correr
# ============================================================
MODO = "ejemplo"           # "ejemplo"  o  "zenodo"

# ---- Solo para MODO="zenodo" -------------------------------
# El dataset en Zenodo tiene 169.513 celulas en total
# 0 = usar las 169.513 completas. Esto NO cabe en el Colab gratuito: se necesita
# activar RAM alta (Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion).
# Si no se tiene RAM alta, se pone aqui un numero menor (p.ej. 60000) para hacer un
# downsample estratificado por tipo celular, pero ten en cuenta que en el
# analisis pre/post-rituximab (seccion 5.2) hay solo 9 pacientes con lupus
# repartidos en varios momentos de tiempo, y el downsample NO estratifica por
# paciente ni por momento: un recorte grande puede dejar a algunos de esos grupos
# con pocas celulas para esa comparacion especifica.
N_CELLS_MAX       = 0        # celulas al convertir el RDS (0 = todas; requiere RAM alta)

# ---- SCENIC: inferencia de red (Parte B) -------------------
SCENIC_DOWNSAMPLE = True     # True = rapido; False = TODAS las celulas B (lento, horas)
SCENIC_N_CELLS    = 2000
SCENIC_N_GENES    = 1500

METODO_GRN = "grnboost2"  # "grnboost2"     = algoritmo real de SCENIC (arboreto)
                          # "sklearn_aprox" = aproximacion NO estandar, solo demo
# Nota: no se recorta la red a mano. 'pyscenic ctx' ya aplica los umbrales
# estandar al construir los modulos (top 50 targets por TF, percentiles 0.75/0.90,
# minimo 20 genes por modulo). Recortar antes solo desviaria del pipeline oficial.

# ---- SCENIC: que hacer si cisTarget no valida ningun regulon (solo MODO="zenodo") ----
# cisTarget filtra el GRN dejando solo los TF-target con motivo de union enriquecido.
# Ese filtro ES el aporte de SCENIC. Si devuelve 0 regulones, seguir sin el produce
# numeros que parecen resultados pero no lo son.
# Este flag SOLO aplica a MODO="zenodo". En MODO="ejemplo" el notebook SIEMPRE
# continua (marcado _SIN_VALIDAR) sin importar este valor: el dataset de juguete
# (500 genes) nunca puede validar regulones por diseno, y ese modo existe para
# mostrar el mecanismo completo, no para producir biologia real.
PERMITIR_REGULONES_SIN_VALIDAR = False   # False = parar con error en zenodo (recomendado para entregables)
                                         # True  = tambien seguir en zenodo, marcando todo como NO VALIDADO

# ---- Columnas de metadatos (None = autodetectar) -----------
# Si la autodeteccion falla te muestra las columnas reales y para; entonces las pones aqui.
COL_COND  = None    # condicion / diagnostico
COL_TIME  = None    # timepoint / tratamiento
COL_CTYPE = None    # tipo celular anotado por los autores

# ---- Volcano pre vs post-rituximab (None = autodetectar) ---
PRE_LABEL  = None
POST_LABEL = None

# ---- Verificacion de descargas -----------------------------
VERIFICAR_CHECKSUMS = True   # comprueba SHA256 de las bases de datos de SCENIC

# ---- Trayectoria (grafica 12) ------------------------------
ROOT_CLUSTER      = None     # cluster raiz del pseudotiempo (str, p.ej. "0"); None = automatico

# ---- Clustering usado para DE / anotacion / trayectoria ----
CLUSTER_KEY       = "leiden_res_0.50"
# ============================================================
assert MODO in ("ejemplo", "zenodo"), f"MODO invalido: {MODO!r}"
assert METODO_GRN in ("grnboost2", "sklearn_aprox"), f"METODO_GRN invalido: {METODO_GRN!r}"
print("MODO:", MODO, "| GRN:", METODO_GRN)
if METODO_GRN == "sklearn_aprox":
    print("AVISO: 'sklearn_aprox' NO es GRNBoost2. Los resultados no son comparables\n"
          "       con la literatura de SCENIC. Usalo solo para demostrar el flujo.")
if MODO == "zenodo" and N_CELLS_MAX == 0:
    print("N_CELLS_MAX=0: se usaran las 169.513 celulas completas del dataset.\n"
          "Esto requiere RAM alta; la celda siguiente lo verifica.")


---
# 2 · Preparar el entorno
Instala las librerías, igual para los dos modos. Si Colab pide reiniciar, hazlo y vuelve a *Ejecutar todas*.


### 2.1 · Memoria disponible

In [ ]:
import psutil, os

ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM: {ram_gb:.1f} GB | CPUs: {os.cpu_count()}")

# El umbral de 20 GB es una estimacion, no una medicion exacta del pico de memoria
# real del pipeline (que depende de cuanto use R al leer el RDS completo antes de
# recortar). Se trata como piso de seguridad: por debajo, es muy probable que el
# proceso muera sin traceback util (Colab mata el kernel en un OOM). Por eso esto
# PARA en vez de solo avisar: seguir a ciegas puede perder horas de computo.
if MODO == "zenodo" and N_CELLS_MAX == 0 and ram_gb < 20:
    raise RuntimeError(
        f"Pediste las 169.513 celulas completas (N_CELLS_MAX=0) pero esta sesion\n"
        f"solo tiene {ram_gb:.1f} GB de RAM. Es muy probable que el paso de conversion\n"
        f"en R (celda 3B.4) o el analisis de scanpy se caigan sin aviso claro.\n\n"
        f"Opciones:\n"
        f"  1) Entorno de ejecucion -> Cambiar tipo de entorno de ejecucion -> RAM alta\n"
        f"     (o Colab Pro/Pro+), y vuelve a ejecutar todo.\n"
        f"  2) Si no tienes acceso a RAM alta, pon un numero en N_CELLS_MAX (celda 1)\n"
        f"     para trabajar con una muestra en vez del dataset completo. No hay un\n"
        f"     numero 'seguro' calculado para esta sesion: es una decision tuya entre\n"
        f"     cuantas celulas conservar y la RAM disponible."
    )


### 2.2 · Instalar scanpy

In [ ]:
# Sin '2>/dev/null': si la instalacion falla queremos verlo, no esconderlo.
import subprocess, sys

def pip_install(paquetes, etiqueta):
    """Instala y PARA si falla, mostrando el error real de pip."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *paquetes]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(
            f"Fallo la instalacion de {etiqueta} (codigo {r.returncode}).\n"
            "Revisa el error de pip arriba. NO sigas: las celdas siguientes\n"
            "fallarian de forma confusa o darian resultados incompletos."
        )
    print(f"{etiqueta}: instalado")

pip_install(["scanpy", "leidenalg", "igraph", "scikit-misc", "fa2-modified"], "scanpy + clustering")

# Verificacion real: que se pueda importar y usar, no solo que pip diga OK
import scanpy as sc, leidenalg, igraph
print("scanpy:", sc.__version__, "| leidenalg:", leidenalg.version)


### 2.3 · Instalar pySCENIC

In [ ]:
pip_install(["pyscenic==0.12.1"], "pySCENIC")

# arboreto 0.1.6 (que trae pyscenic) usa una API de dask.distributed vieja
# ('diagnostics_port' en Server/Worker). Si se deja que pip instale el dask
# mas reciente, arboreto se cae en la celda 6.3 con:
# TypeError: Server.__init__() got an unexpected keyword argument 'diagnostics_port'
pip_install(["dask[distributed]==2023.5.0"], "dask (fijado para arboreto)")

# pySCENIC trae arboreto (GRNBoost2 real) + dask. Verificamos que TODO importe:
# si arboreto no carga, la celda 6.3 no puede correr el algoritmo estandar.
faltan = []
for nombre in ("pyscenic", "ctxcore", "loompy", "arboreto", "dask", "distributed"):
    try:
        __import__(nombre)
    except Exception as e:
        faltan.append(f"  - {nombre}: {type(e).__name__}: {e}")

import pyscenic, numpy as np
print("pyscenic:", pyscenic.__version__, "| numpy:", np.__version__)

if faltan:
    print("\nNo se pudieron importar:\n" + "\n".join(faltan))
    raise RuntimeError(
        "Faltan dependencias de SCENIC.\n"
        "Causa habitual en Colab: pip instalo pyscenic pero el entorno necesita reinicio.\n"
        "Solucion: Entorno de ejecucion -> Reiniciar sesion -> Ejecutar todas."
    )
print("Dependencias de SCENIC: OK (incluye arboreto para GRNBoost2 real)")


### 2.4 · (Opcional) Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR='/content/drive/MyDrive/scanpy_scenic_lupus'; os.makedirs(PROJECT_DIR, exist_ok=True)
except Exception:
    PROJECT_DIR='.'
print("Resultados en:", PROJECT_DIR)


---
# 3 · Carga de datos

Produce el objeto `adata` (crudo, con conteos) según el modo. El análisis y las 12 gráficas de la sección 4 corren después sobre ese mismo `adata`, en los dos modos.


## 3A · [MODO EJEMPLO] Cargar médula ósea
> Solo corre si `MODO=="ejemplo"`.


In [ ]:
if MODO=="ejemplo":
    import scanpy as sc, anndata as ad, numpy as np, pandas as pd, pooch
    sc.settings.verbosity=1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    np.random.seed(0)
    EX = pooch.create(path=pooch.os_cache('scverse_tutorials'),
                      base_url='doi:10.6084/m9.figshare.22716739.v1/')
    EX.load_registry_from_doi()
    samples={'s1d1':'s1d1_filtered_feature_bc_matrix.h5','s1d3':'s1d3_filtered_feature_bc_matrix.h5'}
    adatas={}
    for sid,fn in samples.items():
        a=sc.read_10x_h5(EX.fetch(fn)); a.var_names_make_unique(); adatas[sid]=a
    adata=ad.concat(adatas, label='sample'); adata.obs_names_make_unique()
    col_cond=col_time=col_ctype=None   # no aplican en ejemplo
    print("adata:", adata.shape)


## 3B · [MODO ZENODO] Descargar y convertir los datos de lupus
> Solo corre si `MODO=="zenodo"`. Conversión R→Python en proceso separado (ahorra RAM).


### 3B.1 · Descargar RDS de Zenodo (1.6 GB)

In [ ]:
if MODO == "zenodo":
    import os, hashlib, urllib.request, urllib.error

    RDS = 'RTX_zenodo.RDS'
    URL_RDS = "https://zenodo.org/records/17868028/files/RTX_zenodo.RDS?download=1"
    # Si conoces el SHA256 publicado en Zenodo, ponlo aqui para verificacion fuerte.
    SHA256_RDS = None

    def _es_rds(path):
        """saveRDS escribe gzip (1f 8b) o RDS sin comprimir ('RDX2'/'RDX3')."""
        with open(path, 'rb') as fh:
            m = fh.read(4)
        return m[:2] == b'\x1f\x8b' or m[:3] in (b'RDX', b'RDA')

    def _descargar_rds():
        parcial = RDS + '.part'
        print("Descargando RDS (~1.6 GB)...")
        try:
            with urllib.request.urlopen(URL_RDS, timeout=300) as resp:
                if resp.status != 200:
                    raise RuntimeError(f"HTTP {resp.status} desde Zenodo")
                declarado = resp.headers.get('Content-Length')
                declarado = int(declarado) if declarado else None
                leido = 0
                with open(parcial, 'wb') as fh:
                    while True:
                        trozo = resp.read(1 << 22)
                        if not trozo:
                            break
                        fh.write(trozo); leido += len(trozo)
                        if declarado and leido % (1 << 28) < (1 << 22):
                            print(f"  {leido/1e9:.2f} / {declarado/1e9:.2f} GB")
        except urllib.error.URLError as e:
            if os.path.exists(parcial):
                os.remove(parcial)
            raise RuntimeError(f"No se pudo descargar el RDS desde Zenodo:\n  {e}") from e

        real = os.path.getsize(parcial)
        if declarado is not None and real != declarado:
            os.remove(parcial)
            raise RuntimeError(
                f"Descarga truncada: {real:,} de {declarado:,} bytes. Re-ejecuta la celda."
            )
        if not _es_rds(parcial):
            os.remove(parcial)
            raise RuntimeError(
                "Lo descargado no es un archivo RDS (¿pagina de error de Zenodo?).\n"
                f"Comprueba a mano: {URL_RDS}"
            )
        os.replace(parcial, RDS)

    if not os.path.exists(RDS):
        _descargar_rds()
    elif not _es_rds(RDS):
        # Un RDS corrupto de un intento previo haria fallar Rscript con un error opaco.
        print("El RDS local esta corrupto -> se descarga de nuevo")
        os.remove(RDS); _descargar_rds()

    if SHA256_RDS:
        h = hashlib.sha256()
        with open(RDS, 'rb') as fh:
            for blq in iter(lambda: fh.read(1 << 20), b''):
                h.update(blq)
        if h.hexdigest() != SHA256_RDS:
            raise RuntimeError(f"SHA256 del RDS no coincide:\n  esperado {SHA256_RDS}\n  obtenido {h.hexdigest()}")
        print("SHA256 del RDS: OK")

    print(f"RDS listo: {os.path.getsize(RDS)/1e9:.2f} GB (formato validado)")


### 3B.2 · Instalar SeuratObject en R

In [ ]:
if MODO == "zenodo":
    import subprocess, shutil

    if shutil.which('Rscript') is None:
        raise RuntimeError(
            "No hay R en este entorno. En Colab:  !apt-get install -y r-base\n"
            "El modo 'zenodo' necesita R para leer el objeto Seurat del RDS."
        )

    _r_code = (
        'for (p in c("SeuratObject","Matrix")) '
        'if (!requireNamespace(p, quietly=TRUE)) '
        'install.packages(p, repos="https://cloud.r-project.org"); '
        'ok <- all(sapply(c("SeuratObject","Matrix"), requireNamespace, quietly=TRUE)); '
        'if (!ok) quit(status=1); cat("paquetes R OK\\n")'
    )
    proc = subprocess.run(['Rscript', '-e', _r_code], capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr[-3000:])
        raise RuntimeError(
            "No se pudieron instalar SeuratObject/Matrix en R.\n"
            "Sin ellos la conversion del RDS (celda 3B.4) fallaria mas adelante."
        )


### 3B.3 · Escribir el script de conversión

In [ ]:
%%writefile convert_rds.R
# Convierte el objeto Seurat a formatos que lee Python, sin cargar todo en RAM dos veces.
# Escrito para el RDS de Jang et al.; las suposiciones sobre su estructura se
# comprueban explicitamente para que un dataset distinto falle con un mensaje util.
suppressMessages({library(SeuratObject); library(Matrix)})
n_max <- as.integer(Sys.getenv("N_CELLS_MAX", "60000"))

obj <- readRDS("RTX_zenodo.RDS")
if (!inherits(obj, "Seurat"))
  stop("El RDS no contiene un objeto Seurat, sino: ", paste(class(obj), collapse="/"))

cat("=== ESTRUCTURA ===\n"); print(obj)
cat("\n=== COLUMNAS METADATA ===\n"); print(colnames(obj@meta.data))
meta <- obj@meta.data
cat("\n=== GRUPOS ===\n")
for (col in colnames(meta)) {
  v <- meta[[col]]
  if (is.factor(v) || is.character(v) || (is.numeric(v) && length(unique(v)) < 30))
    if (length(unique(v)) <= 30) { cat("\n[", col, "]\n", sep=""); print(table(v)) }
}

ncells <- ncol(obj)
if (n_max > 0 && n_max < ncells) {
  set.seed(0)
  cc <- grep("celltype|cell_type|cell.type|annotation|ident", colnames(meta),
             ignore.case=TRUE, value=TRUE)
  if (length(cc) > 0) {
    # Downsample estratificado: conserva las proporciones de tipo celular
    grp <- as.character(meta[[cc[1]]]); fr <- n_max / ncells
    idx <- sort(unlist(lapply(split(seq_len(ncells), grp),
                              function(ix) sample(ix, max(1, round(length(ix) * fr))))))
    cat("\n>>> DOWNSAMPLE estratificado por '", cc[1], "': ", ncells, " -> ", length(idx), "\n", sep="")
  } else {
    idx <- sort(sample(ncells, n_max))
    cat("\n>>> DOWNSAMPLE aleatorio:", ncells, "->", length(idx), "\n")
  }
} else {
  idx <- seq_len(ncells); cat("\n>>> TODAS:", ncells, "\n")
}

# --- Assay y counts: comprobamos en vez de asumir ---
ar <- if ("RNA" %in% Assays(obj)) "RNA" else DefaultAssay(obj)
cat("Assay usado:", ar, "de", paste(Assays(obj), collapse=", "), "\n")
counts <- tryCatch(GetAssayData(obj, assay=ar, slot="counts"),
                   error=function(e) tryCatch(GetAssayData(obj, assay=ar, layer="counts"),
                                              error=function(e2) NULL))
if (is.null(counts) || nrow(counts) == 0 || ncol(counts) == 0)
  stop("No se pudieron leer los conteos crudos del assay '", ar, "'. ",
       "SCENIC y el QC necesitan conteos, no datos ya normalizados.")
counts <- counts[, idx]
cat("Matriz de conteos:", nrow(counts), "genes x", ncol(counts), "celulas\n")

Matrix::writeMM(counts, "counts.mtx")
write.csv(data.frame(gene=rownames(counts)), "genes.csv", row.names=FALSE)
write.csv(data.frame(barcode=colnames(counts)), "barcodes.csv", row.names=FALSE)
write.csv(meta[idx, , drop=FALSE], "metadata.csv")

# --- UMAP de los autores: opcional, no siempre existe ---
reds <- Reductions(obj)
if (length(reds) == 0) {
  cat("AVISO: el objeto no tiene reducciones; no se exporta el UMAP de los autores.\n")
} else {
  con_umap <- reds[grepl("umap", tolower(reds))]
  un <- if (length(con_umap) > 0) con_umap[1] else reds[1]
  emb <- Embeddings(obj, un)[idx, , drop=FALSE]
  write.csv(emb, "umap.csv")
  cat("Embedding exportado:", un, "(", ncol(emb), "dimensiones )\n")
}

rm(obj, counts, meta); gc()
cat("\nListo.\n")


### 3B.4 · Ejecutar la conversión (lee el output: ahí están los grupos)

In [ ]:
if MODO == "zenodo":
    import os, subprocess

    entorno = dict(os.environ, N_CELLS_MAX=str(N_CELLS_MAX))
    proc = subprocess.run(['Rscript', 'convert_rds.R'],
                          env=entorno, capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr.strip():
        print("--- stderr de R ---"); print(proc.stderr[-4000:])

    # Si R falla, sin esta comprobacion la celda siguiente rompe con un error opaco
    # ('no such file: counts.mtx') que no dice nada de la causa real.
    if proc.returncode != 0:
        raise RuntimeError(
            f"convert_rds.R fallo (codigo {proc.returncode}). Revisa el stderr de arriba.\n"
            "Causas habituales: falta memoria al leer el RDS de 1.6 GB, o SeuratObject\n"
            "no se instalo bien en la celda 3B.2."
        )

    obligatorios = ['counts.mtx', 'genes.csv', 'barcodes.csv', 'metadata.csv']
    faltan = [f for f in obligatorios if not os.path.exists(f)]
    if faltan:
        raise RuntimeError(
            f"R termino sin error pero no genero: {faltan}\n"
            "Probablemente el objeto Seurat no tiene la estructura esperada\n"
            "(assay 'RNA' con slot 'counts')."
        )
    for f in obligatorios + ['umap.csv']:      # umap.csv es opcional
        if os.path.exists(f):
            print(f"  {f}: {os.path.getsize(f)/1e6:.1f} MB")
        else:
            print(f"  {f}: no generado (el objeto no traia reduccion UMAP; se calculara una nueva)")

    print("\nConversion completada. Lee arriba la seccion '=== GRUPOS ===':\n"
          "ahi estan los valores reales de condicion y timepoint de este dataset.")


### 3B.5 · Reconstruir `adata` (preservando el UMAP de los autores)

In [ ]:
if MODO == "zenodo":
    import os, scipy.io, gc, scanpy as sc, anndata as ad, pandas as pd, numpy as np
    sc.settings.verbosity = 1; sc.settings.set_figure_params(dpi=70, facecolor='white')
    X = scipy.io.mmread('counts.mtx').T.tocsr()
    genes = pd.read_csv('genes.csv')['gene'].astype(str).values
    bc = pd.read_csv('barcodes.csv')['barcode'].astype(str).values
    meta = pd.read_csv('metadata.csv', index_col=0)
    adata = ad.AnnData(X=X, obs=meta, var=pd.DataFrame(index=genes))
    adata.obs_names = bc

    if os.path.exists('umap.csv'):                     # el UMAP de los autores es opcional
        umap = pd.read_csv('umap.csv', index_col=0)
        adata.obsm['X_umap_authors'] = umap.values
        print(f"UMAP de los autores conservado: {umap.shape[1]} dimensiones")
        del umap
    else:
        print("Sin UMAP de los autores; la seccion 4 calculara uno nuevo.")
    del X, meta; gc.collect()

    def _resumen_columnas():
        """Lista las columnas de obs con sus valores, para poder elegir a mano."""
        lineas = []
        for c in adata.obs.columns:
            v = adata.obs[c]
            n = v.nunique(dropna=True)
            if n <= 12:
                lineas.append(f"    {c!r:38s} ({n:2d} valores) -> {sorted(map(str, v.dropna().unique()))}")
            else:
                lineas.append(f"    {c!r:38s} ({n:4d} valores, continuo/ID)")
        return "\n".join(lineas)

    def resolver_columna(etiqueta, override, claves, obligatoria):
        """Devuelve la columna elegida. Nunca 'None en silencio':
        si es obligatoria y no aparece, para y te ensena las columnas reales."""
        if override is not None:
            if override not in adata.obs.columns:
                raise KeyError(
                    f"La columna {override!r} que fijaste para '{etiqueta}' no existe.\n"
                    f"Columnas disponibles:\n{_resumen_columnas()}"
                )
            print(f"  {etiqueta:6s}: {override!r}  (fijada a mano)")
            return override

        candidatas = [c for c in adata.obs.columns if any(k in c.lower() for k in claves)]
        if len(candidatas) == 1:
            print(f"  {etiqueta:6s}: {candidatas[0]!r}  (autodetectada)")
            return candidatas[0]
        if len(candidatas) > 1:
            print(f"  {etiqueta:6s}: {candidatas[0]!r}  (autodetectada; habia varias "
                  f"{candidatas} -> si no es la correcta fijala en la celda 1)")
            return candidatas[0]

        msg = (f"No se encontro ninguna columna para '{etiqueta}'.\n"
               f"Este notebook se escribio para el RDS de Jang et al.; otro dataset\n"
               f"puede nombrar sus columnas de otra forma.\n"
               f"Solucion: elige la columna correcta de esta lista y ponla en la celda 1\n"
               f"como COL_{etiqueta.upper()} = 'nombre_de_columna'.\n"
               f"Columnas disponibles:\n{_resumen_columnas()}")
        if obligatoria:
            raise KeyError(msg)
        print(f"  {etiqueta:6s}: NO encontrada (opcional)\n{msg}\n")
        return None

    print("Resolviendo columnas de metadatos:")
    col_cond  = resolver_columna('cond',  COL_COND,
                                 ['disease', 'condition', 'group', 'sle', 'status', 'diagnosis'],
                                 obligatoria=False)
    col_time  = resolver_columna('time',  COL_TIME,
                                 ['time', 'visit', 'day', 'week', 'treatment', 'point'],
                                 obligatoria=False)
    # col_ctype es obligatoria: sin ella no se pueden seleccionar las celulas B (secciones 5 y 6)
    col_ctype = resolver_columna('ctype', COL_CTYPE,
                                 ['celltype', 'cell_type', 'cell.type', 'annotation', 'ident'],
                                 obligatoria=True)
    print()
    print(adata)


### 3B.6 · Guardar h5ad y liberar disco

In [ ]:
if MODO=="zenodo":
    import os, gc
    adata.write_h5ad(f"{PROJECT_DIR}/lupus_rituximab.h5ad")
    if os.path.exists('counts.mtx'): os.remove('counts.mtx')
    gc.collect(); print("Guardado.")


---
# 4 · Análisis y las 12 gráficas

Este bloque corre en los dos modos sobre `adata` y genera las 12 gráficas.


### 4.0 · Preparar marcadores según el modo
Se filtra los genes que sí existen en los datos (evita errores si falta alguno).


In [ ]:
import scanpy as sc, numpy as np, pandas as pd

def present(md_dict):
    out={k:[g for g in v if g in adata.var_names] for k,v in md_dict.items()}
    return {k:v for k,v in out.items() if v}

if MODO=="ejemplo":
    MARKERS = present({
        'CD14+ Mono':['FCN1','CD14'],'CD16+ Mono':['TCF7L2','FCGR3A','LYN'],
        'cDC2':['CST3','COTL1','LYZ','CLEC10A','FCER1A'],
        'Erythroblast':['MKI67','HBA1','HBB'],'Proerythroblast':['CDK6','SYNGR1','HBM','GYPA'],
        'NK':['GNLY','NKG7','CD247','TYROBP','KLRG1'],
        'Naive CD20+ B':['MS4A1','IL4R','IGHD','FCRL1','IGHM'],
        'Plasma cells':['MZB1','HSP90B1','PRDM1','IGKC','JCHAIN'],
        'CD4+ T':['CD4','IL7R','TRBC2'],'CD8+ T':['CD8A','CD8B','GZMK','CCL5','GZMB'],
        'T naive':['LEF1','CCR7','TCF7'],'pDC':['IL3RA','COBLL1','TCF4'],
    })
else:
    MARKERS = present({
        'T CD4':['CD3D','CD4','IL7R'],'T CD8':['CD3D','CD8A','GZMK'],
        'B':['MS4A1','CD79A','CD79B'],'NK':['GNLY','NKG7','KLRD1'],
        'Mono':['CD14','LYZ','FCGR3A'],'DC':['FCER1A','CST3'],
        'Plasma':['MZB1','JCHAIN','IGHG1'],
    })
print("Marcadores:", list(MARKERS.keys()))


### 4.1 · [Gráfica 1] Violines de QC
Número de genes por célula, conteos totales y % de conteos mitocondriales.


In [ ]:
adata.var['mt']  =adata.var_names.str.startswith('MT-')
adata.var['ribo']=adata.var_names.str.startswith(('RPS','RPL'))
adata.var['hb']  =adata.var_names.str.contains('^HB[^(P)]')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], inplace=True, log1p=True)
sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'],
             jitter=0.4, multi_panel=True)


### 4.2 · [Gráfica 2] Scatter de QC coloreado
Conteos totales vs. genes detectados, color = % mitocondrial.


In [ ]:
sc.pl.scatter(adata, 'total_counts', 'n_genes_by_counts', color='pct_counts_mt')

# Filtrado: en ejemplo (crudo) filtramos + dobletes; en zenodo ya viene curado
if MODO=="ejemplo":
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)
    sc.pp.scrublet(adata, batch_key='sample', random_state=0)
    print("Dobletes:", int(adata.obs['predicted_doublet'].sum()))
else:
    sc.pp.filter_genes(adata, min_cells=3)   # limpieza suave de genes


### 4.3 · [Gráfica 3] Selección de características: normalizado vs no
Guardamos los conteos crudos, normalizamos + log1p, y marcamos los genes altamente variables (HVG).


In [ ]:
adata.layers['counts']=adata.X.copy()
sc.pp.normalize_total(adata); sc.pp.log1p(adata)
batch = 'sample' if 'sample' in adata.obs.columns else None
sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key=batch)
sc.pl.highly_variable_genes(adata)   # muestra normalizado vs no


### 4.4 · [Gráfica 4] PCA: PC1/PC2 y PC3/PC4
Coloreado por grupo y por % mitocondrial. Incluye la varianza explicada por componente.


In [ ]:
sc.tl.pca(adata, random_state=0)
color_group = 'sample' if 'sample' in adata.obs.columns else (col_cond or 'pct_counts_mt')
# color y dimensions se emparejan por posicion (zip): repetimos para PC1/2 y PC3/4
sc.pl.pca(adata,
          color=[color_group, color_group, 'pct_counts_mt', 'pct_counts_mt'],
          dimensions=[(0,1), (2,3), (0,1), (2,3)], ncols=2, size=3)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)


### 4.5 · [Gráfica 5] Grafo de vecinos más cercanos
Construimos el grafo kNN y el UMAP; dibujamos las aristas del grafo sobre el UMAP.


In [ ]:
sc.pp.neighbors(adata, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.pl.umap(adata, color=color_group, edges=True, edges_width=0.05,
           title='Grafo de vecinos mas cercanos')


### 4.6 · Clustering Leiden (base para el resto)
Tres resoluciones. El resto de gráficas usa `CLUSTER_KEY` (por defecto res 0.50).


In [ ]:
for res in [0.02, 0.5, 2.0]:
    sc.tl.leiden(adata, key_added=f'leiden_res_{res:4.2f}', resolution=res,
                 flavor='igraph', n_iterations=2, random_state=0)
    print(f"res {res}: {adata.obs[f'leiden_res_{res:4.2f}'].nunique()} clusters")


### 4.7 · [Gráfica 6] Filtrado visto con UMAP (métricas de QC)
Las métricas de calidad proyectadas sobre el UMAP, para ver si algún cluster es artefacto.


In [ ]:
qc_cols=['n_genes_by_counts','total_counts','pct_counts_mt']
if 'doublet_score' in adata.obs.columns: qc_cols.append('doublet_score')
sc.pl.umap(adata, color=qc_cols, ncols=2, size=3)


### 4.8 · [Gráfica 7] Anotación manual: números en cada grupo
Clusters con su número encima. En zenodo se muestra también la anotación de los autores.


In [ ]:
sc.pl.umap(adata, color=CLUSTER_KEY, legend_loc='on data', title='Clusters (numeros)')

if MODO=="ejemplo":
    cluster_to_celltype={'0':'Lymphocytes','1':'Monocytes','2':'Erythroid','3':'B Cells'}
    adata.obs['cell_type']=(adata.obs['leiden_res_0.02'].map(cluster_to_celltype)
                            .fillna('Unknown').astype('category'))
    sc.pl.umap(adata, color='cell_type', legend_loc='on data')
elif col_ctype:
    sc.pl.umap(adata, color=col_ctype, legend_loc='right margin',
               title='Anotacion de los autores')


### 4.9 · [Gráfica 8] Patrones de expresión de marcadores + dotplot
Los patrones se generan solo con el nombre del gen, el símbolo; no hace falta secuencia. scanpy busca el gen en la matriz y colorea según su expresión.


In [ ]:
# Dotplot de marcadores por cluster
sc.pl.dotplot(adata, MARKERS, groupby=CLUSTER_KEY, standard_scale='var')

# Patrones de expresion de algunos marcadores sobre el UMAP
genes_umap=[g for gs in MARKERS.values() for g in gs][:6]
sc.pl.umap(adata, color=genes_umap, ncols=3, size=3)


### 4.10 · Expresión diferencial (base para gráficas 9-11)
Test de Wilcoxon: genes que definen cada cluster.

Antes del test excluimos los genes housekeeping (mitocondriales `MT-`, ribosomales `RPS`/`RPL` y hemoglobina `HB`). Dominan el ranking como ruido técnico, apareciendo como "diferenciales" en casi todos los clusters sin ser marcadores biológicos útiles. El DE se calcula sobre `adata_de`, la copia sin esos genes; `adata` queda intacto.


In [ ]:
# Excluir housekeeping: dominan el DE como ruido y no son marcadores reales
hk = (adata.var_names.str.startswith(('MT-','RPS','RPL','MRPS','MRPL')) |
      adata.var_names.str.contains('^HB[^(P)]'))
adata_de = adata[:, ~hk].copy()
print(f"Genes housekeeping excluidos del DE: {int(hk.sum())} | genes usados: {adata_de.n_vars}")
sc.tl.rank_genes_groups(adata_de, groupby=CLUSTER_KEY, method='wilcoxon')
print("Expresion diferencial calculada sobre", CLUSTER_KEY)


### 4.11 · [Gráfica 9] Dotplot de genes diferencialmente expresados

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_de, groupby=CLUSTER_KEY,
                                standard_scale='var', n_genes=5)


### 4.12 · [Gráfica 10] Expresión diferencial: tabla completa + top-100 + gráficas

Se exportan dos archivos, sacados de la misma tabla:

- `DE_completo_<modo>.csv`: todos los genes evaluados por clúster (ya sin housekeeping), con puntaje, cambio de expresión (`logfoldchanges`) y significancia estadística (`pvals_adj`). Nada se descarta aquí.
- `top100_DE_<modo>.csv`: los genes realmente significativos (`pvals_adj < 0.05`) de cada clúster, hasta un máximo de 100. 

Rellenar hasta 100 con genes que no pasan el corte estadístico daría un listado con el mismo aspecto en todos los clústeres, aunque buena parte no signifique nada, casi imposible de distinguir a simple vista de los genes que sí importan. Filtrando primero por significancia y recortando después se cumple el pedido de entrega, hasta 100 genes por clúster, sin fingir que hay más señal biológica de la que en realidad hay.


In [ ]:
de_all = sc.get.rank_genes_groups_df(adata_de, group=None)
de_all['significativo'] = de_all['pvals_adj'] < 0.05

# Tabla completa: todos los genes evaluados por cluster (significativos y no).
# Es la referencia: no depende de ninguna cuota fija.
out_csv = f"{PROJECT_DIR}/DE_completo_{MODO}.csv"
de_all.to_csv(out_csv, index=False)

# "Top 100": los genes REALMENTE significativos de cada cluster, hasta un techo
# de 100 por cluster. El 100 es un MAXIMO, no una cuota a rellenar: se filtra
# por significancia PRIMERO y se recorta despues, en ese orden.
top100 = (de_all[de_all['significativo']]
          .groupby('group', group_keys=False)
          .head(100))
out_csv_top100 = f"{PROJECT_DIR}/top100_DE_{MODO}.csv"
top100.to_csv(out_csv_top100, index=False)

n_sig_por_cluster = de_all.groupby('group')['significativo'].sum().astype(int)
print(f"Guardado: {out_csv}  ({len(de_all):,} filas = todos los genes evaluados x "
      f"{de_all['group'].nunique()} clusters)")
print(f"Guardado: {out_csv_top100}  ({len(top100):,} filas: hasta 100 genes "
      f"REALMENTE significativos por cluster, sin relleno)")
print("\nGenes significativos (pvals_adj < 0.05) por cluster:")
print(n_sig_por_cluster.to_string())

incompletos = n_sig_por_cluster[(n_sig_por_cluster < 100) & (n_sig_por_cluster > 0)]
if len(incompletos) > 0:
    print(f"\n{len(incompletos)} cluster(es) con MENOS de 100 genes significativos: "
          "top100_DE trae solo los que hay, no se rellena para llegar a 100.")
    print(incompletos.to_string())

sin_significativos = n_sig_por_cluster[n_sig_por_cluster == 0]
if len(sin_significativos) > 0:
    print(f"\nAVISO: {len(sin_significativos)} cluster(es) sin NINGUN gen significativo "
          f"(pvals_adj < 0.05): {list(sin_significativos.index)}")
    print(f"No apareceran en top100_DE_{MODO}.csv: no hay nada valido que listar ahi. "
          "Siguen presentes en DE_completo con su ranking real, por si quieres revisarlos.")

# Vista rapida en pantalla
display(top100.groupby('group').head(5))

# Grafica de rankings (top genes por cluster)
sc.pl.rank_genes_groups(adata_de, n_genes=20, sharey=False)


### 4.13 · [Gráfica 11] Heatmap de expresión diferencial
Con pocos genes por cluster y ejes intercambiados (`swap_axes`) para que los nombres se lean.


In [ ]:
sc.pl.rank_genes_groups_heatmap(adata_de, n_genes=3, groupby=CLUSTER_KEY,
                                standard_scale='var', show_gene_labels=True,
                                swap_axes=True, figsize=(12,14))


### 4.14 · [Gráfica 12] Trayectoria (PAGA + pseudotiempo)
Reconstruye linajes: PAGA conecta los clusters según su similitud, y el pseudotiempo (DPT) ordena las células a lo largo del linaje. Necesita una célula raíz (`ROOT_CLUSTER`); si se deja en `None`, se elige sola.


In [ ]:
# PAGA: grafo de conexiones entre clusters
sc.tl.paga(adata, groups=CLUSTER_KEY)
sc.pl.paga(adata, color=CLUSTER_KEY, title='PAGA: conexiones entre clusters')

# UMAP inicializado con PAGA (respeta la topologia de los linajes)
sc.tl.umap(adata, init_pos='paga', random_state=0)

# Pseudotiempo: elegir raiz
import numpy as np
root = ROOT_CLUSTER if ROOT_CLUSTER is not None else adata.obs[CLUSTER_KEY].value_counts().index[-1]
adata.uns['iroot'] = int(np.flatnonzero(adata.obs[CLUSTER_KEY]==str(root))[0])
sc.tl.dpt(adata)
print("Cluster raiz del pseudotiempo:", root)
sc.pl.umap(adata, color=[CLUSTER_KEY,'dpt_pseudotime'], legend_loc='on data', size=3)


---
# 5 · [MODO ZENODO] Figuras del artículo de lupus
> Solo corre si `MODO=="zenodo"`. Usa la anotación de los autores.


### 5.1 · Figura 1: Subtipos de células B


In [ ]:
# Definida siempre (aunque en modo 'ejemplo' no se use): la celda 6.2 tambien la llama,
# asi el criterio de "que es una celula B" vive en UN solo sitio.
B_KW = ['naive b', 'transitional', 'memory b', 'switched', 'abc',
        'plasmablast', 'plasma', 'b cell', 'b_cell']

def seleccionar_celulas_B(adata, col_ctype, min_celulas=50):
    """Subconjunto de celulas B segun la anotacion de los autores.

    Los nombres de subtipo son los de Zenodo. Si el dataset usa otra
    nomenclatura, el filtro no encuentra nada: en ese caso paramos y mostramos
    las etiquetas reales, en vez de seguir con un objeto vacio.
    """
    etiquetas = adata.obs[col_ctype].astype(str)
    es_B = etiquetas.str.lower().str.contains('|'.join(B_KW), regex=True)
    n = int(es_B.sum())
    if n < min_celulas:
        raise ValueError(
            f"El filtro de celulas B encontro {n} celulas (minimo esperado: {min_celulas}).\n"
            f"Se buscaron estas palabras clave en {col_ctype!r}: {B_KW}\n"
            f"Etiquetas reales en el dataset:\n"
            + "\n".join(f"    {e!r}: {c:,}" for e, c in etiquetas.value_counts().items())
            + "\n\nAjusta B_KW en esta celda para que coincida con la nomenclatura de tus datos."
        )
    sub = adata[es_B].copy()
    print(f"Celulas B: {sub.n_obs:,} de {adata.n_obs:,} ({100*sub.n_obs/adata.n_obs:.1f}%)")
    return sub

if MODO == "zenodo":
    adata_B = seleccionar_celulas_B(adata, col_ctype)
    print(adata_B.obs[col_ctype].value_counts())
    sc.pl.umap(adata_B, color=col_ctype, size=8, title='Subtipos de celulas B (Lupus)')


### 5.2 · Figura 2: Volcano pre vs post-rituximab

Las etiquetas de *timepoint* se detectan solas a partir de los valores reales de la columna. Si la detección es ambigua (por ejemplo, hay varios momentos post-tratamiento), la celda se detiene y muestra los valores disponibles para que elijas: se fijan en `PRE_LABEL` / `POST_LABEL`, en la celda 1, sin tocar el código.


In [ ]:
if MODO == "zenodo":
    CELLTYPE_SUBSET = None      # p.ej. 'Memory B' para restringir el volcano a un tipo celular

    if col_time is None:
        raise RuntimeError(
            "No hay columna de timepoint, asi que no se puede comparar pre vs post.\n"
            "Fija COL_TIME en la celda 1 con la columna correcta (la celda 3B.5 las listo)."
        )

    valores = adata.obs[col_time].astype(str)
    disponibles = sorted(valores.unique())
    conteos = valores.value_counts()
    _lista = "\n".join(f"    {v!r}: {conteos[v]:,} celulas" for v in disponibles)

    def _elegir(termino, claves, excluir):
        """Busca UN valor que encaje. Si hay 0 o mas de 1, para y muestra las opciones."""
        cand = [v for v in disponibles
                if any(k in v.lower() for k in claves)
                and not any(x in v.lower() for x in excluir)]
        if len(cand) == 1:
            return cand[0]
        motivo = "no encaja ninguno" if not cand else f"encajan varios: {cand}"
        raise RuntimeError(
            f"No se pudo determinar automaticamente el valor de '{termino}' ({motivo}).\n"
            f"Valores reales de {col_time!r}:\n{_lista}\n\n"
            f"Elige el que corresponda y ponlo en la celda 1:\n"
            f"    PRE_LABEL  = '...'   # antes del rituximab\n"
            f"    POST_LABEL = '...'   # despues del rituximab"
        )

    # 'pre' se excluye al buscar post para que 'Pretreatment' no cuente como post-tratamiento
    pre  = PRE_LABEL  if PRE_LABEL  is not None else _elegir(
        'PRE',  ['pre', 'baseline', 'before', 'screening'], ['post'])
    post = POST_LABEL if POST_LABEL is not None else _elegir(
        'POST', ['post', 'after', 'follow'], ['pre-', 'pretreat'])

    for etiqueta, valor in (('PRE_LABEL', pre), ('POST_LABEL', post)):
        if valor not in disponibles:
            raise ValueError(f"{etiqueta}={valor!r} no existe en {col_time!r}.\n"
                             f"Valores reales:\n{_lista}")
    PRE_LABEL, POST_LABEL = pre, post
    print(f"Comparacion: {POST_LABEL!r}  vs  {PRE_LABEL!r}  (columna {col_time!r})")

    ad_de = adata
    if CELLTYPE_SUBSET and col_ctype:
        ad_de = ad_de[ad_de.obs[col_ctype].astype(str) == CELLTYPE_SUBSET]
        print(f"Restringido a {CELLTYPE_SUBSET!r}: {ad_de.n_obs:,} celulas")
    ad_de = ad_de[ad_de.obs[col_time].astype(str).isin([PRE_LABEL, POST_LABEL])].copy()

    n_por_grupo = ad_de.obs[col_time].astype(str).value_counts()
    print(n_por_grupo)
    if n_por_grupo.min() < 30:
        raise ValueError(
            f"Uno de los grupos tiene solo {n_por_grupo.min()} celulas.\n"
            "Con tan pocas, el test de Wilcoxon no es fiable y el volcano no seria interpretable.\n"
            "Sube N_CELLS_MAX, o quita CELLTYPE_SUBSET, o elige otros timepoints."
        )

    sc.tl.rank_genes_groups(ad_de, groupby=col_time, groups=[POST_LABEL],
                            reference=PRE_LABEL, method='wilcoxon')
    de = sc.get.rank_genes_groups_df(ad_de, group=POST_LABEL)
    print(f"Genes evaluados: {len(de):,}")


In [ ]:
if MODO=="zenodo":
    import matplotlib.pyplot as plt, numpy as np
    d=de.dropna(subset=['logfoldchanges','pvals_adj']).copy()
    d['nlp']=-np.log10(d['pvals_adj'].clip(lower=1e-300))
    up=(d['logfoldchanges']>1)&(d['pvals_adj']<0.05); dn=(d['logfoldchanges']<-1)&(d['pvals_adj']<0.05)
    plt.figure(figsize=(8,6))
    plt.scatter(d['logfoldchanges'],d['nlp'],s=6,c='lightgray')
    plt.scatter(d.loc[up,'logfoldchanges'],d.loc[up,'nlp'],s=8,c='#c0392b',label='Up')
    plt.scatter(d.loc[dn,'logfoldchanges'],d.loc[dn,'nlp'],s=8,c='#2e5f9a',label='Down')
    for _,r in d[up|dn].nlargest(15,'nlp').iterrows(): plt.text(r['logfoldchanges'],r['nlp'],r['names'],fontsize=7)
    plt.axvline(0,color='k',lw=.5); plt.axhline(-np.log10(0.05),color='k',ls='--',lw=.5)
    plt.xlabel(f'logFC ({POST_LABEL} - {PRE_LABEL})'); plt.ylabel('-log10 p'); plt.legend()
    plt.title('Volcano post vs pre-rituximab'); plt.tight_layout(); plt.show()


---
# 6 · Parte B: Redes regulatorias (SCENIC)

GRNBoost2 → cisTarget → AUCell. Corre en ambos modos; la matriz de entrada cambia (celda 6.2).


### 6.0 · Compatibilidad de pySCENIC con numpy moderno

pySCENIC 0.12.1, de 2022, usa `np.object`, `np.bool`, etc. Numpy eliminó esos alias en la versión 1.24, así que en un Colab actual pySCENIC puede caerse con `AttributeError: module 'numpy' has no attribute 'object'`.

Esta celda restaura los alias, que no son más que `object`, `bool`, `int`..., de dos formas: en memoria para este proceso, y mediante un archivo `.pth` para que también los tenga el subproceso que lanza `pyscenic ctx`.

Esto evita que el proceso se caiga, pero no hace que aparezcan regulones: son dos problemas distintos. Si cisTarget termina bien pero devuelve 0 regulones, la causa es biológica o del dataset (pocos genes, TFs sin motivos en la base hg38), no este alias. La celda 6.4 distingue ambos casos.



In [ ]:
# Restaurar los alias que numpy>=1.24 elimino. NO se modifica ningun archivo de pySCENIC.
import numpy as np, site, os, sys, subprocess

ALIAS = {'object': object, 'bool': bool, 'int': int, 'float': float, 'str': str}

# (a) En memoria: cubre este proceso y los hijos por fork (multiprocessing en Linux/Colab)
restaurados = [n for n in ALIAS if not hasattr(np, n)]
for n, t in ALIAS.items():
    if not hasattr(np, n):
        setattr(np, n, t)

# (b) Archivo .pth: python lo ejecuta al arrancar, asi el subproceso 'pyscenic ctx'
#     (que es un interprete nuevo, sin nuestra memoria) tambien tiene los alias.
_linea = ("import numpy as _np; "
          "[setattr(_np, _n, _t) for _n, _t in "
          "[('object', object), ('bool', bool), ('int', int), ('float', float), ('str', str)] "
          "if not hasattr(_np, _n)]\n")
_dirs = []
if hasattr(site, 'getusersitepackages'):
    _dirs.append(site.getusersitepackages())
if hasattr(site, 'getsitepackages'):
    _dirs.extend(site.getsitepackages())

_pth_ok = False
for _dir in _dirs:
    try:
        os.makedirs(_dir, exist_ok=True)
        with open(os.path.join(_dir, 'zzz_numpy_alias_pyscenic.pth'), 'w') as fh:
            fh.write(_linea)
        _pth_ok = True
        break
    except OSError:
        continue

print(f"numpy {np.__version__} | alias restaurados en memoria: {restaurados or 'ninguno (no hacian falta)'}")

# Verificamos que el subproceso REALMENTE los tenga (no damos por hecho que el .pth funciono)
_r = subprocess.run([sys.executable, "-c",
                     "import numpy; print(hasattr(numpy,'object') and hasattr(numpy,'bool'))"],
                    capture_output=True, text=True)
_sub_ok = _r.stdout.strip() == "True"
print(f"archivo .pth escrito: {_pth_ok} | subproceso con alias: {_sub_ok}")

if not _sub_ok:
    print("\nAVISO: el subproceso no tiene los alias. Si 'pyscenic ctx' (celda 6.4) falla con\n"
          "AttributeError sobre np.object, la alternativa es fijar numpy<1.24 y reiniciar:\n"
          "    !pip install 'numpy<1.24'\n"
          "Si cisTarget corre sin error, este aviso no afecta al resultado.")


### 6.1 · Descargar bases de datos de SCENIC

In [ ]:
import os, hashlib, urllib.request, urllib.error

os.makedirs('scenic_data', exist_ok=True)

# SHA256 verificados de las bases de datos de cisTarget.
# Si aertslab publica una version nueva, el hash cambiara y esta celda parara:
# eso es lo correcto (te enteras de que los datos cambiaron), no un fallo.
# Para re-fijar: pon el hash nuevo aqui despues de comprobar que el cambio es esperado.
CHECKSUMS = {
    'motifs.tbl':          '81eb754118e27e854974301b1400fcf519489f8be5249239671fb288cb501c31',
    'hg38_rankings.feather':'9c4026a3a8e25fe07cf96749644e2ca028b787410829b30b9932574dc6e78bdb',
    'expr_mat_tiny.loom':  'ca57894cc828488d7aeb3ca58ad76a637f265c502688856d45a73d39f9483b4c',
    'test_TFs_tiny.txt':   '6fa3c3c273f4da4e7ebb4189b21a3126abf9d899f619a5dfb28155e937524d1f',
    'allTFs_hg38.txt':     None,   # sin fijar: se imprime el hash para que lo fijes tu
}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blq in iter(lambda: fh.read(chunk), b''):
            h.update(blq)
    return h.hexdigest()

def parece_html(path):
    """Un error 404/403 suele guardarse como pagina HTML con nombre de .feather."""
    with open(path, 'rb') as fh:
        inicio = fh.read(512).lstrip().lower()
    return inicio.startswith(b'<!doctype html') or inicio.startswith(b'<html')

def descargar_verificado(nombre, url, destino):
    esperado = CHECKSUMS.get(nombre)

    # Un archivo ya presente puede estar truncado de un intento anterior: se verifica igual.
    if os.path.exists(destino):
        if not VERIFICAR_CHECKSUMS or esperado is None:
            print(f"  {nombre}: {os.path.getsize(destino)/1e6:.1f} MB (ya estaba)")
            return
        obtenido = sha256(destino)
        if obtenido == esperado:
            print(f"  {nombre}: {os.path.getsize(destino)/1e6:.1f} MB  SHA256 OK")
            return
        print(f"  {nombre}: el archivo local NO coincide con el hash fijado -> se rebaja")
        os.remove(destino)

    # Descarga a .part y renombrado al final: nunca dejamos un archivo truncado
    # con el nombre definitivo (esa es la causa clasica de fallos raros mas adelante).
    parcial = destino + '.part'
    print(f"  Descargando {nombre} ...")
    try:
        with urllib.request.urlopen(url, timeout=120) as resp:
            if resp.status != 200:
                raise RuntimeError(f"HTTP {resp.status} al descargar {nombre}")
            declarado = resp.headers.get('Content-Length')
            declarado = int(declarado) if declarado else None
            with open(parcial, 'wb') as fh:
                while True:
                    trozo = resp.read(1 << 20)
                    if not trozo:
                        break
                    fh.write(trozo)
    except urllib.error.URLError as e:
        if os.path.exists(parcial):
            os.remove(parcial)
        raise RuntimeError(f"No se pudo descargar {nombre} desde {url}\n  {e}") from e

    real = os.path.getsize(parcial)
    if declarado is not None and real != declarado:
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: descarga truncada ({real:,} de {declarado:,} bytes).\n"
            "Vuelve a ejecutar la celda; suele ser un corte de red temporal."
        )
    if parece_html(parcial):
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: el servidor devolvio una pagina HTML, no el archivo.\n"
            f"URL probablemente caida o movida: {url}"
        )

    obtenido = sha256(parcial)
    if esperado is None:
        print(f"    (sin hash fijado) SHA256 = {obtenido}")
    elif VERIFICAR_CHECKSUMS and obtenido != esperado:
        os.remove(parcial)
        raise RuntimeError(
            f"{nombre}: SHA256 no coincide.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}\n"
            "O la descarga se corrompio (re-ejecuta), o el archivo de origen cambio.\n"
            "Si el cambio es esperado, actualiza CHECKSUMS con el hash nuevo."
        )

    os.replace(parcial, destino)
    print(f"  {nombre}: {real/1e6:.1f} MB  SHA256 {'OK' if esperado else 'registrado'}")

COM = {
 'motifs.tbl': 'https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl',
 'hg38_rankings.feather': 'https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/refseq_r80/mc_v10_clust/gene_based/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
}
if MODO == "ejemplo":
    COM['expr_mat_tiny.loom'] = 'https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/expr_mat_tiny.loom'
    COM['test_TFs_tiny.txt']  = 'https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/test_TFs_tiny.txt'
else:
    COM['allTFs_hg38.txt'] = 'https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt'

for fn, url in COM.items():
    descargar_verificado(fn, url, f'scenic_data/{fn}')

print("Bases de datos de SCENIC listas y verificadas.")


### 6.2 · Preparar matriz + TFs según el modo

In [ ]:
import pandas as pd, numpy as np

if MODO == "ejemplo":
    import loompy
    with loompy.connect('scenic_data/expr_mat_tiny.loom') as ds:
        ex_matrix = pd.DataFrame(ds[:, :].T, index=ds.ca['CellID'], columns=ds.ra['Gene'])
    tf_names = pd.read_csv('scenic_data/test_TFs_tiny.txt', header=None).iloc[:, 0].tolist()
    N_EST = 500
else:
    import scanpy as sc
    if 'adata_B' not in globals():          # por si se corre la seccion 6 sin la 5
        adata_B = seleccionar_celulas_B(adata, col_ctype)
    ad_s = adata_B.copy()
    if SCENIC_DOWNSAMPLE and ad_s.n_obs > SCENIC_N_CELLS:
        sc.pp.subsample(ad_s, n_obs=SCENIC_N_CELLS, random_state=0)
        print(f"Downsample a {ad_s.n_obs:,} celulas B")
    else:
        print(f"Sin downsample: {ad_s.n_obs:,} celulas B (puede tardar horas)")

    sc.pp.highly_variable_genes(ad_s, n_top_genes=min(SCENIC_N_GENES, ad_s.n_vars - 1))
    ad_s = ad_s[:, ad_s.var.highly_variable].copy()
    Xd = ad_s.X.toarray() if hasattr(ad_s.X, 'toarray') else np.asarray(ad_s.X)
    ex_matrix = pd.DataFrame(Xd, index=ad_s.obs_names.astype(str),
                             columns=ad_s.var_names.astype(str))
    all_tfs = pd.read_csv('scenic_data/allTFs_hg38.txt', header=None).iloc[:, 0].tolist()
    tf_names = [t for t in all_tfs if t in ex_matrix.columns]
    N_EST = 200

print(f"Matriz: {ex_matrix.shape[0]:,} celulas x {ex_matrix.shape[1]:,} genes | TFs: {len(tf_names)}")

# cisTarget necesita bastantes genes para encontrar motivos enriquecidos.
# Con muy pocos, el paso 6.4 devolvera 0 regulones casi con seguridad.
if ex_matrix.shape[1] < 1000:
    print(f"\nAVISO: solo {ex_matrix.shape[1]} genes en la matriz. cisTarget probablemente\n"
          "no validara ningun regulon (necesita suficientes targets por TF para detectar\n"
          "enriquecimiento de motivos). Es lo esperable con el dataset 'tiny' de ejemplo.")


### 6.3 · Inferencia de la red (GRNBoost2)

Primer paso de SCENIC: para cada gen se entrena un modelo que predice su expresión a partir de la de los factores de transcripción. Los TF que más ayudan a predecir un gen quedan enlazados a él. El resultado es una lista de aristas TF-gen con una importancia asociada.

Se usa `arboreto.grnboost2`, la implementación de referencia de SCENIC.



In [ ]:
import os, time, pandas as pd, numpy as np

tfs_in = [t for t in tf_names if t in ex_matrix.columns]
if not tfs_in:
    raise ValueError(
        "Ningun factor de transcripcion de la lista aparece en la matriz de expresion.\n"
        "Sin TFs no hay red que inferir. Suele significar que los nombres de gen no\n"
        "coinciden (p.ej. symbols vs Ensembl IDs)."
    )
print(f"TFs presentes en la matriz: {len(tfs_in)} de {len(tf_names)}")

t0 = time.time()

if METODO_GRN == "grnboost2":
    from arboreto.algo import grnboost2

    def _correr_grnboost2():
        """Intenta con un cliente dask explicito; si no, deja que arboreto cree el suyo.
        Ambas rutas ejecutan EL MISMO algoritmo: no hay degradacion silenciosa."""
        try:
            from distributed import Client, LocalCluster
            n_w = max(1, min(4, os.cpu_count() or 1))
            cluster = LocalCluster(n_workers=n_w, threads_per_worker=1, processes=True)
            cliente = Client(cluster)
            print(f"  dask: {n_w} workers")
            try:
                return grnboost2(expression_data=ex_matrix, tf_names=tfs_in,
                                 client_or_address=cliente, seed=42, verbose=True)
            finally:
                cliente.close(); cluster.close()
        except Exception as e:
            print(f"  cliente dask explicito no disponible ({type(e).__name__}: {e})")
            print("  reintentando con el scheduler interno de arboreto...")
            return grnboost2(expression_data=ex_matrix, tf_names=tfs_in,
                             seed=42, verbose=True)

    try:
        adjacencies = _correr_grnboost2()
    except Exception as e:
        raise RuntimeError(
            f"GRNBoost2 fallo: {type(e).__name__}: {e}\n\n"
            "Causa habitual: incompatibilidad entre arboreto 0.1.6 y la version de dask del entorno.\n"
            "Opciones:\n"
            "  1) !pip install 'dask[distributed]==2023.5.0' y reiniciar el entorno\n"
            "  2) METODO_GRN = 'sklearn_aprox' en la celda 1 (aproximacion NO estandar,\n"
            "     valida solo para demostrar el flujo, no para resultados publicables)\n"
            "No se cambia de metodo automaticamente: seria cambiar el algoritmo sin que te enteres."
        ) from e

else:
    # Aproximacion explicitamente elegida. Mismo espiritu (importancia por gradient boosting)
    # pero NO es GRNBoost2: sin early-stopping ni el esquema de muestreo de arboreto.
    from sklearn.ensemble import GradientBoostingRegressor
    print("Aproximacion sklearn (NO estandar). Esto puede tardar bastante...")
    X_tfs = ex_matrix[tfs_in].values
    targets = [g for g in ex_matrix.columns if g not in tfs_in]
    registros = []
    for i, tg in enumerate(targets):
        y = ex_matrix[tg].values
        if y.std() == 0:
            continue
        gbm = GradientBoostingRegressor(n_estimators=N_EST, max_depth=3, random_state=42)
        gbm.fit(X_tfs, y)
        for tf, imp in zip(tfs_in, gbm.feature_importances_):
            if imp > 0:
                registros.append({'TF': tf, 'target': tg, 'importance': float(imp)})
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(targets)}")
    adjacencies = pd.DataFrame(registros)

if adjacencies.empty:
    raise RuntimeError("La inferencia de red no produjo ninguna arista TF-gen.")

adjacencies = adjacencies.sort_values('importance', ascending=False).reset_index(drop=True)

# La red se guarda SIN recortar: 'pyscenic ctx' aplica los umbrales estandar
# (top 50 targets/TF, percentiles 0.75-0.90, min 20 genes por modulo) al generar
# los modulos. Recortar aqui se desviaria del pipeline oficial.
os.makedirs('scenic_data', exist_ok=True)
adjacencies.to_csv('scenic_data/adjacencies.tsv', sep='\t', index=False)

por_tf = adjacencies.groupby('TF').size()
n_targets_posibles = ex_matrix.shape[1] - len(tfs_in)
print(f"\nMetodo: {METODO_GRN} | {time.time()-t0:.0f}s")
print(f"Aristas TF-gen: {len(adjacencies):,}")
print(f"TFs con targets: {por_tf.size} | targets por TF: "
      f"mediana {por_tf.median():.0f}, min {por_tf.min()}, max {por_tf.max()}")

# Diagnostico de densidad. Si CADA TF apunta a CASI TODOS los genes, las importancias
# no estan discriminando: cisTarget elegira sus top-50 por TF practicamente al azar.
if n_targets_posibles > 0:
    densidad = por_tf.median() / n_targets_posibles
    print(f"Densidad (mediana targets/TF entre genes disponibles): {densidad:.1%}")
    if densidad > 0.9:
        print(
            "\nAVISO: la red es casi un grafo completo. Las importancias apenas distinguen\n"
            "unos targets de otros, asi que los modulos que construya cisTarget seran\n"
            "practicamente arbitrarios. Con METODO_GRN='sklearn_aprox' esto es lo esperable."
        )


### 6.4 · cisTarget

In [ ]:
import loompy, numpy as np, pandas as pd, subprocess, os

LOOM = 'scenic_data/expr_mat_tiny.loom' if MODO == "ejemplo" else 'scenic_data/expr_bcells.loom'
if MODO == "zenodo":
    if os.path.exists(LOOM):
        os.remove(LOOM)          # loompy.create falla si el archivo ya existe
    loompy.create(LOOM, ex_matrix.T.values,
                  {'Gene': np.array(ex_matrix.columns)},
                  {'CellID': np.array(ex_matrix.index)})

cmd = ['pyscenic', 'ctx', 'scenic_data/adjacencies.tsv', 'scenic_data/hg38_rankings.feather',
       '--annotations_fname', 'scenic_data/motifs.tbl',
       '--expression_mtx_fname', LOOM,
       '--output', 'scenic_data/regulons.csv',
       '--num_workers', '2']
print("Ejecutando:", " ".join(cmd), "\n")
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.stdout.strip():
    print(proc.stdout[-2500:])
if proc.stderr.strip():
    print("--- log de cisTarget ---"); print(proc.stderr[-2500:])

# CASO 1: cisTarget se cayo. Aqui SI puede ser el problema de np.object (celda 6.0).
if proc.returncode != 0:
    pista = ""
    if 'np.object' in proc.stderr or "has no attribute 'object'" in proc.stderr:
        pista = ("\nEs el problema de numpy: re-ejecuta la celda 6.0, y si persiste,\n"
                 "    !pip install 'numpy<1.24'   y reinicia el entorno.")
    raise RuntimeError(f"'pyscenic ctx' fallo (codigo {proc.returncode}).{pista}")

# CASO 2: cisTarget corrio bien. Cuantos regulones valido?
try:
    df_reg = pd.read_csv('scenic_data/regulons.csv', index_col=[0, 1], header=[0, 1])
    n_regulons = len(df_reg)
except Exception:
    n_regulons = 0

print(f"\nRegulones validados por motivos: {n_regulons}")
if n_regulons == 0:
    print(
        "\ncisTarget termino sin error pero no valido ningun regulon.\n"
        "Esto NO es el bug de numpy: es que ningun TF tiene motivos de union\n"
        "enriquecidos entre sus targets. Causas tipicas:\n"
        "  - dataset demasiado pequeno (el 'tiny' de ejemplo tiene 500 genes)\n"
        "  - TFs cuyos motivos no estan en la base hg38 (p.ej. BRF1, de Pol III)\n"
        "  - nombres de gen que no casan con la base (symbols vs Ensembl)\n"
        "  - la red del paso anterior era demasiado densa o demasiado pobre\n"
        "La celda 6.5 decide que hacer con esto."
    )


### 6.5 · AUCell: actividad de cada regulón en cada célula

AUCell puntúa, célula a célula, cuán arriba están los genes de cada regulón en su ranking de expresión.

En `MODO="zenodo"`, por defecto el notebook se detiene aquí si cisTarget no validó ningún regulón. Una versión anterior seguía adelante construyendo "regulones" directamente desde las aristas del GRN, sin el filtro por motivos, que es justo lo que distingue a SCENIC de una simple red de coexpresión. No es un riesgo teórico: en una ejecución real guardada en `runs/`, cisTarget devolvió 0 regulones, ese atajo se activó, y el `auc_matrix.csv` resultante contenía 20 "regulones" cuyos AUC iban todos de 0,0136 a 0,0414, puro ruido. El "top 10 por actividad" separaba 0,0242 de 0,0239, diferencias en la cuarta decimal presentadas como un ranking, y sobre eso se calcularon un UMAP y 9 clusters que parecían resultados. Si aun así se quiere ver el flujo completo sin datos válidos en zenodo, se pone `PERMITIR_REGULONES_SIN_VALIDAR = True` en la celda 1.

En `MODO="ejemplo"` el notebook continúa siempre, automáticamente, sin que se tenga que tocar nada: el dataset, 500 genes, nunca va a validar regulones por motivos, así que exigir esa validación ahí impediría mostrar el mecanismo completo de la demostración. La salida queda marcada `_SIN_VALIDAR`, y el notebook lo repite en pantalla. No es un resultado biológico real, es una ilustración de cómo se ve el flujo hasta el final.


In [ ]:
from pyscenic.aucell import aucell as pyscenic_aucell
from ctxcore.genesig import GeneSignature
import ast, pandas as pd

signatures = []

# En MODO='ejemplo' el dataset de juguete (500 genes) nunca puede validar
# regulones por motivos, una limitacion del tamano del dataset y no un
# fallo. Para que el modo de demostracion muestre el mecanismo completo, se
# continua siempre ahi sin depender de PERMITIR_REGULONES_SIN_VALIDAR (que sigue
# controlando, sin cambios, el comportamiento estricto en MODO='zenodo').
continuar_sin_validar = PERMITIR_REGULONES_SIN_VALIDAR or (MODO == "ejemplo")

if n_regulons > 0:
    # Camino normal: regulones validados por motivos de union
    REGULONES_VALIDADOS = True
    dfc = pd.read_csv('scenic_data/regulons.csv', header=[0, 1], index_col=[0, 1])
    dfc.columns = [' '.join(c).strip() for c in dfc.columns]
    dfc = dfc.reset_index()
    tgt = next((c for c in dfc.columns if 'TargetGenes' in c), None)
    tfc = next((c for c in dfc.columns if c in ('TF', 'level_0')), None)
    if tgt is None or tfc is None:
        raise RuntimeError(
            f"regulons.csv no tiene el formato esperado. Columnas: {list(dfc.columns)}"
        )
    for _, row in dfc.iterrows():
        try:
            gs = [t[0] for t in ast.literal_eval(str(row[tgt]))]
        except (ValueError, SyntaxError):
            gs = []
        if gs:
            signatures.append(GeneSignature(name=f"{row[tfc]}(+)",
                                            gene2weight={g: 1.0 for g in gs}))
    if not signatures:
        raise RuntimeError(
            f"cisTarget reporto {n_regulons} regulones pero ninguno tenia genes target legibles.\n"
            "Revisa scenic_data/regulons.csv."
        )

elif not continuar_sin_validar:
    # Parada deliberada (solo alcanzable en MODO='zenodo'). Seguir aqui produce
    # numeros con forma de resultado que no lo son.
    raise RuntimeError(
        "cisTarget no valido ningun regulon, asi que NO hay nada que puntuar con AUCell.\n"
        "\n"
        "El notebook para aqui a proposito. Construir los regulones desde el GRN sin el\n"
        "filtro por motivos elimina justo el paso que diferencia a SCENIC de una red de\n"
        "coexpresion: el resultado son puntuaciones casi identicas entre si (ruido) que\n"
        "luego se grafican como si fueran biologia.\n"
        "\n"
        "Estas en MODO='zenodo'. Que hacer:\n"
        "  - Revisa el diagnostico de la celda 6.3 (que los nombres de gen casen con la\n"
        "    base hg38, y que la densidad de la red no sea degenerada).\n"
        "  - Solo para demostrar el flujo: PERMITIR_REGULONES_SIN_VALIDAR = True en la\n"
        "    celda 1. Las salidas quedaran marcadas como NO VALIDADAS."
    )

else:
    # Se continua porque: (a) MODO='ejemplo' (el dataset de juguete nunca valida
    # por diseno, y este modo existe para mostrar el mecanismo completo), o
    # (b) el usuario activo PERMITIR_REGULONES_SIN_VALIDAR en MODO='zenodo'.
    REGULONES_VALIDADOS = False
    causa = ("MODO='ejemplo': el dataset de juguete no puede validar por diseno"
             if MODO == "ejemplo" and not PERMITIR_REGULONES_SIN_VALIDAR
             else "PERMITIR_REGULONES_SIN_VALIDAR=True (opt-in explicito en zenodo)")
    print("=" * 70)
    print(f"ATENCION: regulones SIN validacion por motivos. Causa: {causa}")
    print("Esto NO es SCENIC completo. Los resultados no son interpretables como")
    print("actividad regulatoria real y no deben entregarse como tal.")
    print("=" * 70)
    for tf, grp in adjacencies.groupby('TF'):
        signatures.append(GeneSignature(name=f"{tf}(+)",
                                        gene2weight=dict(zip(grp['target'], grp['importance']))))

SUFIJO = "" if REGULONES_VALIDADOS else "_SIN_VALIDAR"
tam = [len(s.genes) for s in signatures]
print(f"\nFirmas: {len(signatures)} | genes por firma: "
      f"mediana {int(pd.Series(tam).median())}, min {min(tam)}, max {max(tam)}")

auc_matrix = pyscenic_aucell(ex_matrix, signatures, num_workers=1)
print("AUCell:", auc_matrix.shape)

# Un rango de AUC practicamente plano indica que las firmas no discriminan celulas.
rango = auc_matrix.values.max() - auc_matrix.values.min()
print(f"AUC: min {auc_matrix.values.min():.4f} | max {auc_matrix.values.max():.4f} "
      f"| rango {rango:.4f}")
if rango < 0.05:
    print("\nAVISO: el rango de AUC es minusculo; las firmas apenas separan unas celulas\n"
          "de otras. Cualquier cluster o ranking calculado sobre esto sera ruido.")


### 6.6 · Visualizar regulones

In [ ]:
import scanpy as sc

asc = sc.AnnData(X=auc_matrix.values,
                 obs=pd.DataFrame(index=auc_matrix.index.astype(str)),
                 var=pd.DataFrame(index=auc_matrix.columns.astype(str)))
sc.pp.neighbors(asc, random_state=42)
sc.tl.umap(asc, random_state=42)
sc.tl.leiden(asc, flavor='igraph', n_iterations=2, random_state=42)

# El titulo lleva la procedencia: un grafico sin contexto acaba en una presentacion
# sin el aviso que lo acompanaba en la consola.
etiqueta = "regulones validados por motivos" if REGULONES_VALIDADOS else "SIN VALIDAR: no interpretable"
sc.pl.umap(asc, color='leiden', title=f'Actividad de regulones ({MODO}): {etiqueta}')

salida = f"{PROJECT_DIR}/scenic_auc_{MODO}{SUFIJO}.csv"
auc_matrix.to_csv(salida)
print("Guardado:", salida)
if not REGULONES_VALIDADOS:
    print("\nRecuerda: el sufijo _SIN_VALIDAR indica que estos scores NO pasaron el\n"
          "filtro por motivos de cisTarget. No los entregues como resultado de SCENIC.")


### 6.7 · Procedencia: qué se ejecutó realmente

Deja constancia del método usado y de si los regulones pasaron la validación por motivos. 


In [ ]:
# ============================================================
#   PROCEDENCIA: que se ejecuto realmente
# ============================================================
import datetime, sys, importlib

def _ver(mod):
    try:
        return importlib.import_module(mod).__version__
    except Exception:
        return "no disponible"

apto = REGULONES_VALIDADOS and METODO_GRN == "grnboost2"

print("=" * 62)
print("  PROCEDENCIA DEL ANALISIS SCENIC")
print("=" * 62)
print(f"  Fecha              : {datetime.datetime.now():%Y-%m-%d %H:%M}")
print(f"  Modo de datos      : {MODO}")
print(f"  Inferencia de red  : {METODO_GRN}"
      f"{'  (algoritmo estandar de SCENIC)' if METODO_GRN == 'grnboost2' else '  (APROXIMACION NO ESTANDAR)'}")
print(f"  Aristas del GRN    : {len(adjacencies):,}")
print(f"  Regulones cisTarget: {n_regulons}")
print(f"  Validado x motivos : {'SI' if REGULONES_VALIDADOS else 'NO'}")
print(f"  Matriz AUCell      : {auc_matrix.shape[0]:,} celulas x {auc_matrix.shape[1]} regulones")
print(f"  Rango de AUC       : {auc_matrix.values.min():.4f} - {auc_matrix.values.max():.4f}")
print("-" * 62)
print(f"  Python {sys.version.split()[0]} | numpy {_ver('numpy')} | "
      f"scanpy {_ver('scanpy')} | pyscenic {_ver('pyscenic')}")
print("=" * 62)

if apto:
    print("  APTO PARA ENTREGA: algoritmo estandar + regulones validados.")
else:
    motivos = []
    if METODO_GRN != "grnboost2":
        motivos.append("la red no se infirio con GRNBoost2")
    if not REGULONES_VALIDADOS:
        motivos.append("los regulones no pasaron el filtro por motivos")
    print("  NO APTO PARA ENTREGA como resultado de SCENIC, porque "
          + " y ".join(motivos) + ".")
    if MODO == "ejemplo":
        print("  Esto es lo ESPERADO en MODO='ejemplo': el dataset de juguete (500 genes)")
        print("  nunca puede validar regulones por diseno. Sirve para demostrar el")
        print("  mecanismo completo, no como resultado biologico. Usa MODO='zenodo'")
        print("  para un analisis real.")
    else:
        print("  Sirve para demostrar el flujo del pipeline, no como resultado biologico.")
print("=" * 62)
